# 第0章 环境准备

在开始学习 SQL 之前，我们需要先准备好开发环境。本章将指导你完成以下步骤：

1. 安装 MySQL 数据库
2. 安装 Python 和必要的库
3. 创建示例数据库
4. 在 Jupyter Notebook 中连接 MySQL

---

## 1. 安装 MySQL

### Windows 用户

1. 访问 [MySQL 官网下载页面](https://dev.mysql.com/downloads/installer/)
2. 下载 MySQL Installer（推荐下载较大的完整安装包）
3. 运行安装程序，选择 **Developer Default** 或 **Server only**
4. 设置 root 用户密码（请牢记！）
5. 完成安装

### macOS 用户

```bash
# 使用 Homebrew 安装
brew install mysql

# 启动 MySQL 服务
brew services start mysql

# 或者使用 dmg 安装包
# 从 https://dev.mysql.com/downloads/mysql/ 下载
```

### Linux (Ubuntu) 用户

```bash
sudo apt update
sudo apt install mysql-server
sudo systemctl start mysql
sudo systemctl enable mysql
```

### 验证安装

打开终端/命令行，输入：

```bash
mysql --version
```

如果显示版本号，说明安装成功。

## 2. 安装 Python 依赖

本教程需要以下 Python 库：

| 库名 | 用途 |
|------|------|
| `pymysql` | 连接 MySQL 数据库 |
| `pandas` | 数据展示（表格格式） |
| `jupyter` | 运行 Notebook |

安装命令：

In [ ]:
# 安装必要的 Python 库
!pip install pymysql pandas jupyter -q

## 3. 创建示例数据库

本教程提供了一个初始化脚本 `setup_database.py`，它会自动创建示例数据库 `shop_db` 并填充测试数据。

### 运行方式

在终端中执行：

```bash
# 先修改 setup_database.py 中的数据库密码
python setup_database.py
```

### 或者在 Notebook 中运行：

In [ ]:
# 运行数据库初始化脚本
# 注意：请先确保 MySQL 服务已启动，并修改了正确的密码
!python setup_database.py

## 4. 在 Notebook 中连接 MySQL

接下来我们创建一个通用的数据库连接函数，在后续章节中都会用到。

In [ ]:
import pymysql
import pandas as pd

# 数据库连接配置
DB_CONFIG = {
    "host": "localhost",
    "user": "root",
    "password": "your_password",  # ← 修改为你的 MySQL 密码
    "database": "shop_db",
    "charset": "utf8mb4",
}

def get_connection():
    """获取数据库连接"""
    return pymysql.connect(**DB_CONFIG)

def run_query(sql, params=None):
    """
    执行 SQL 查询并返回 DataFrame
    
    参数:
        sql: SQL 查询语句
        params: 查询参数（可选）
    
    返回:
        pandas.DataFrame: 查询结果
    """
    conn = get_connection()
    try:
        df = pd.read_sql(sql, conn, params=params)
        return df
    finally:
        conn.close()

def execute_sql(sql, params=None):
    """
    执行非查询 SQL（INSERT/UPDATE/DELETE）
    
    参数:
        sql: SQL 语句
        params: 参数（可选）
    
    返回:
        int: 受影响的行数
    """
    conn = get_connection()
    try:
        cursor = conn.cursor()
        affected = cursor.execute(sql, params)
        conn.commit()
        return affected
    finally:
        conn.close()

### 测试连接

让我们执行一个简单的查询来验证连接是否正常：

In [ ]:
# 测试连接 - 查看所有表
run_query("SHOW TABLES")

In [ ]:
# 查看商品表的数据
run_query("SELECT * FROM products LIMIT 5")

In [ ]:
# 查看各表的记录数
for table in ['categories', 'products', 'customers', 'employees', 'orders', 'order_items']:
    result = run_query(f"SELECT COUNT(*) AS cnt FROM {table}")
    print(f"{table:15s} → {result['cnt'].iloc[0]} 条记录")

## 5. 示例数据库结构预览

我们的示例数据库 `shop_db` 是一个简化的电商系统，包含以下 6 张表：

```
┌─────────────┐     ┌─────────────┐     ┌──────────────┐
│ categories  │     │  products   │     │ order_items  │
│─────────────│     │─────────────│     │──────────────│
│ category_id │◄────│ category_id │     │ item_id      │
│ category_name│    │ product_id  │◄────│ product_id   │
│ description │     │ product_name│     │ order_id     │
└─────────────┘     │ price       │     │ quantity     │
                    │ stock       │     │ unit_price   │
                    └─────────────┘     └──────────────┘
                                               │
                    ┌─────────────┐            │
                    │   orders    │◄───────────┘
                    │─────────────│
                    │ order_id    │
                    │ customer_id │◄────┐
                    │ employee_id │◄──┐ │
                    │ order_date  │   │ │
                    │ status      │   │ │
                    │ total_amount│   │ │
                    └─────────────┘   │ │
                                      │ │
┌─────────────┐     ┌─────────────┐   │ │
│  customers  │     │  employees  │   │ │
│─────────────│     │─────────────│   │ │
│ customer_id │────►│ employee_id │───┘ │
│ customer_name│    │ employee_name│    │
│ email       │     │ department  │    │
│ phone       │     │ salary      │    │
│ city        │     │ hire_date   │    │
└─────────────┘     │ manager_id  │    │
                    └─────────────┘    │
```

### 各表字段说明

| 表名 | 说明 | 主要字段 |
|------|------|----------|
| `categories` | 商品分类 | category_id, category_name, description |
| `products` | 商品信息 | product_id, product_name, category_id, price, stock |
| `customers` | 客户信息 | customer_id, customer_name, email, phone, city |
| `employees` | 员工信息 | employee_id, employee_name, department, salary, hire_date |
| `orders` | 订单信息 | order_id, customer_id, employee_id, order_date, status |
| `order_items` | 订单明细 | item_id, order_id, product_id, quantity, unit_price |

## 6. 常见问题

### Q: 连接数据库时报错 `Access denied`
A: 检查用户名和密码是否正确。如果忘记 root 密码，可以重置：
```bash
sudo mysql_secure_installation
```

### Q: 报错 `Can't connect to MySQL server`
A: 检查 MySQL 服务是否已启动：
```bash
# Windows
net start mysql

# macOS
brew services start mysql

# Linux
sudo systemctl start mysql
```

### Q: 中文显示乱码
A: 确保连接时指定了 `charset=utf8mb4`，这已经在我们的配置中设置好了。

---

## ✅ 环境检查清单

- [ ] MySQL 已安装并启动
- [ ] Python 已安装
- [ ] pymysql、pandas、jupyter 已安装
- [ ] 示例数据库 shop_db 已创建
- [ ] 能成功执行 `run_query("SELECT 1")`

全部完成后，我们就可以开始学习 SQL 了！🎉

**下一章：[01_数据库基础概念](01_数据库基础概念.ipynb)**